# Project 1: Social Media Engagement & Sentiment Analysis

Run the cells top to bottom in Google Colab. Cells marked **YOU EDIT** need your input.
Everything the submission needs is saved into the `submission/` folder at the end.

## Step 1 · Load the data and keep a backup

In [ ]:
import os, shutil, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
os.makedirs("submission", exist_ok=True)

# ---- YOU EDIT ----------------------------------------------------------
CSV_PATH = "your_dataset.csv"   # replace with your uploaded filename
# If your column names differ, map them to the names this notebook uses, e.g.
# RENAME = {"likeCount": "likes", "replyCount": "comments", "textDisplay": "text", "publishedAt": "timestamp"}
RENAME = {}
# -------------------------------------------------------------------------

try:  # Colab: show an upload dialog if the file is not there yet
    from google.colab import files
    if not os.path.exists(CSV_PATH):
        uploaded = files.upload()
        CSV_PATH = list(uploaded.keys())[0]
except ImportError:
    pass

shutil.copy(CSV_PATH, "ORIGINAL_backup_" + os.path.basename(CSV_PATH))  # untouched copy
raw = pd.read_csv(CSV_PATH)
df = raw.rename(columns=RENAME).copy()
has = lambda *cols: all(c in df.columns for c in cols)
print("Loaded:", df.shape)
df.head()

## Step 2 · What is one record, and what does each column mean?

**YOU EDIT:** write one sentence here, e.g. *"One record = one social media post by one user on one platform."*
Then fix the `meanings` text below so every column has an accurate description.

In [ ]:
# Guessed meanings for the common columns: CHECK each one against the README / your data.
meanings = {
    "post_id": "Unique identifier of the post",
    "timestamp": "Date and time of posting",
    "platform": "Social media platform",
    "user": "Account that posted",
    "followers_count": "Followers of the account",
    "verified": "Whether the account is verified",
    "text": "Post text",
    "hashtags": "Hashtags used in the post",
    "country": "Country of the post/user",
    "likes": "Number of likes",
    "retweets_shares": "Number of retweets/shares",
    "comments": "Number of comments",
    "sentiment": "Sentiment label of the post",
}
def kind(s):
    if pd.api.types.is_numeric_dtype(s): return "Numerical"
    if pd.api.types.is_datetime64_any_dtype(s): return "Date/time"
    return "Text/Categorical"

data_dictionary = pd.DataFrame({
    "column": df.columns,
    "dtype_as_loaded": [str(t) for t in df.dtypes],
    "type_guess": [kind(df[c]) for c in df.columns],
    "example_value": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    "meaning": [meanings.get(c, "TODO: describe this column") for c in df.columns],
})
data_dictionary.to_csv("submission/data_dictionary.csv", index=False)
data_dictionary

## Step 3 · Inspect before cleaning (record count, types, missing values, duplicates)

In [ ]:
print("Rows, columns:", df.shape)
display(df.dtypes)
display(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
if has("post_id"):
    print("Duplicate post_id values:", df["post_id"].duplicated().sum())
display(df.describe(include="all").T)

# Look for numbers stored as text and impossible values BEFORE cleaning
for c in ["followers_count", "likes", "retweets_shares", "comments"]:
    if c in df.columns and not pd.api.types.is_numeric_dtype(df[c]):
        print(f"'{c}' is stored as text (e.g. commas). Will convert.")
    if c in df.columns and pd.api.types.is_numeric_dtype(df[c]) and (df[c] < 0).any():
        print(f"'{c}' has {(df[c] < 0).sum()} negative values. Will review.")

## Step 4 · Clean (every change is logged)

Rule used here: **do not silently delete rows.** Impossible values become *missing* (NaN) and are reported.
A blank metric may mean "not collected", not zero.

In [ ]:
log = []
def note(step, n, detail=""):
    log.append({"step": step, "rows_or_cells_affected": int(n), "detail": detail})

# 1) Counts stored as text ("1,234") -> numbers
num_cols = [c for c in ["followers_count", "likes", "retweets_shares", "comments"] if c in df.columns]
for c in num_cols:
    before_na = df[c].isna().sum()
    df[c] = pd.to_numeric(df[c].astype(str).str.replace(",", "", regex=False).str.strip(), errors="coerce")
    note(f"convert {c} to numeric", df[c].isna().sum() - before_na, "values that could not be parsed became NaN")

# 2) Negative counts are impossible -> NaN
for c in num_cols:
    neg = (df[c] < 0).sum()
    df.loc[df[c] < 0, c] = np.nan
    note(f"negative {c} set to NaN", neg)

# 3) Timestamp -> datetime (invalid -> NaT)
if has("timestamp"):
    before_na = df["timestamp"].isna().sum()
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    note("parse timestamp", df["timestamp"].isna().sum() - before_na, "invalid dates became NaT")

# 4) Standardise text categories (trim spaces, consistent case)
for c in ["platform", "country", "sentiment"]:
    if c in df.columns:
        df[c] = df[c].astype("string").str.strip().str.title()
if has("text"):
    df["text"] = df["text"].astype("string").str.strip()

# 5) Duplicates (logged, then removed)
n = df.duplicated().sum(); df = df.drop_duplicates(); note("drop exact duplicate rows", n)
if has("post_id"):
    n = df["post_id"].duplicated().sum(); df = df.drop_duplicates(subset="post_id", keep="first")
    note("drop repeated post_id (keep first)", n)

cleaning_log = pd.DataFrame(log)
cleaning_log.to_csv("submission/cleaning_log.csv", index=False)
display(cleaning_log)
print("Rows after cleaning:", len(df))
print(df.isna().sum()[lambda s: s > 0])

## Step 5 · Summary statistics

Two required statistics are produced below. Only analyses the columns can support are run.

In [ ]:
metrics = [c for c in ["likes", "retweets_shares", "comments"] if c in df.columns]

# Statistic 1: descriptive statistics of the engagement numbers
desc = df[metrics + (["followers_count"] if has("followers_count") else [])].describe().T
desc["median"] = df[desc.index].median()
display(desc)

# Statistic 2: group counts AND medians (never compare totals without group sizes)
if has("platform"):
    by_platform = df.groupby("platform")[metrics].agg(["count", "mean", "median"]).round(1)
    display(by_platform)
    by_platform.to_csv("submission/stats_by_platform.csv")
if has("sentiment"):
    display(df["sentiment"].value_counts(dropna=False).to_frame("records"))

# Are a few posts dominating the averages?
if has("likes"):
    top10 = df["likes"].nlargest(max(1, int(len(df) * 0.10))).sum() / df["likes"].sum()
    print(f"Mean likes: {df['likes'].mean():.1f} | Median likes: {df['likes'].median():.1f}")
    print(f"Top 10% of posts account for {top10:.0%} of all likes")
    print("Correlation likes vs comments (Spearman):",
          round(df[["likes", "comments"]].corr(method="spearman").iloc[0, 1], 3) if has("comments") else "n/a")

# Engagement per follower (only if follower counts exist; this is NOT reach-based engagement rate)
if has("followers_count") and metrics:
    d = df[df["followers_count"] > 0].copy()
    d["engagement_per_follower"] = d[metrics].sum(axis=1, min_count=1) / d["followers_count"]
    if has("platform"):
        display(d.groupby("platform")["engagement_per_follower"].agg(["count", "median"]).round(4))

## Step 6 · Charts (at least two are required)

In [ ]:
# Chart 1: records per sentiment
if has("sentiment"):
    ax = df["sentiment"].value_counts().plot(kind="bar", color="#4C78A8")
    plt.title("Record count by sentiment"); plt.xlabel("Sentiment"); plt.ylabel("Number of records")
    plt.xticks(rotation=0); plt.tight_layout(); plt.savefig("submission/chart1_sentiment_counts.png", dpi=150); plt.show()

# Chart 2: median likes by platform, with group sizes in the labels
if has("platform", "likes"):
    g = df.groupby("platform")["likes"].agg(["median", "count"]).sort_values("median", ascending=False)
    ax = g["median"].plot(kind="bar", color="#F58518")
    ax.set_xticklabels([f"{p}\n(n={int(n)})" for p, n in zip(g.index, g["count"])], rotation=0)
    plt.title("Median likes by platform"); plt.ylabel("Median likes"); plt.xlabel("")
    plt.tight_layout(); plt.savefig("submission/chart2_median_likes_by_platform.png", dpi=150); plt.show()

# Chart 3: likes vs comments (log scale because a few posts are huge)
if has("likes", "comments"):
    d = df[(df["likes"] > 0) & (df["comments"] > 0)]
    plt.scatter(d["likes"], d["comments"], alpha=0.35, s=12)
    plt.xscale("log"); plt.yscale("log")
    plt.title("Likes vs comments (log scales)"); plt.xlabel("Likes"); plt.ylabel("Comments")
    plt.tight_layout(); plt.savefig("submission/chart3_likes_vs_comments.png", dpi=150); plt.show()

# Chart 4: distribution of likes (shows skew)
if has("likes"):
    df["likes"].dropna().plot(kind="hist", bins=40, color="#54A24B")
    plt.title("Distribution of likes"); plt.xlabel("Likes"); plt.ylabel("Number of posts")
    plt.tight_layout(); plt.savefig("submission/chart4_likes_distribution.png", dpi=150); plt.show()

# Chart 5: posts per month (only if timestamps are valid)
if has("timestamp") and df["timestamp"].notna().sum() > 0:
    df.dropna(subset=["timestamp"]).set_index("timestamp").resample("MS").size().plot()
    plt.title("Posts per month"); plt.xlabel("Month"); plt.ylabel("Posts")
    plt.tight_layout(); plt.savefig("submission/chart5_posts_per_month.png", dpi=150); plt.show()

## Step 7 · Business interpretation

Fill the table below in your report. Use the evidence printed above (copy the real numbers).

| | Your answer |
|---|---|
| **Finding 1** | What pattern do you actually see? |
| **Evidence 1** | Which count/statistic/chart supports it? (include group sizes) |
| **Finding 2** | |
| **Evidence 2** | |
| **Action (the one recommendation)** | What should a marketer test next, using a consistent metric? |
| **Limitation** | What can this dataset not tell us? (e.g. unknown provenance, no reach/impressions, platform sizes differ) |

Template: *"In this dataset, category X has a higher median number of comments than category Y (n = ...). The brand could test more posts in category X and compare results using a consistent metric."*
Do **not** say a content type is "best" because its total likes are higher.

## Step 8 · Scaling experiment (this is a workload test, not new evidence)

In [ ]:
RUN_2M = False   # set True only if your Colab session has enough RAM; it can be slow

def analysis(d):
    t = time.time()
    if has("platform") and metrics:
        d.groupby("platform")[metrics].agg(["mean", "median"])
    if has("sentiment"):
        d["sentiment"].value_counts()
    return time.time() - t

targets = [2_000, 20_000, 200_000] + ([2_000_000] if RUN_2M else [])
rows = []
for target in targets:
    reps = int(np.ceil(target / len(df)))
    big = pd.concat([df] * reps, ignore_index=True).head(target)
    big.to_csv("scaling_test.csv", index=False)
    size_mb = os.path.getsize("scaling_test.csv") / 1e6
    t0 = time.time(); reloaded = pd.read_csv("scaling_test.csv"); load_s = time.time() - t0
    rows.append({"rows": target, "file_MB": round(size_mb, 1), "load_seconds": round(load_s, 3),
                 "analysis_seconds": round(analysis(big), 3),
                 "memory_MB": round(big.memory_usage(deep=True).sum() / 1e6, 1)})
    del big, reloaded
scaling = pd.DataFrame(rows)
scaling.to_csv("submission/scaling_results.csv", index=False)
display(scaling)
os.remove("scaling_test.csv")

Write 2–3 sentences: which step got slower, how memory grew, and whether anything was inconvenient.
Pandas has no universal row count where it "fails". Spark becomes worth investigating when data volume,
processing time, or distributed storage needs outgrow one machine, not for a 2,000-row file.

## Step 9 · Short ML preview (conceptual demo only)

Each demo avoids leakage: no target metric is used to build its own predictor, and predictive work uses a train/test split.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import classification_report, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# CLASSIFICATION: predict the sentiment label from post text
if has("text", "sentiment"):
    d = df.dropna(subset=["text", "sentiment"])
    Xtr, Xte, ytr, yte = train_test_split(d["text"].astype(str), d["sentiment"], test_size=0.25,
                                          random_state=42, stratify=d["sentiment"])
    vec = TfidfVectorizer(max_features=3000, ngram_range=(1, 2), stop_words="english")
    clf = LogisticRegression(max_iter=1000).fit(vec.fit_transform(Xtr), ytr)
    print(classification_report(yte, clf.predict(vec.transform(Xte))))

In [ ]:
# CLUSTERING: group posts with similar (scaled) numeric profiles
feat = [c for c in ["followers_count", "likes", "retweets_shares", "comments"] if c in df.columns]
if len(feat) >= 2:
    d = df.dropna(subset=feat).copy()
    Z = StandardScaler().fit_transform(np.log1p(d[feat]))
    d["cluster"] = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(Z)
    display(d.groupby("cluster")[feat].median().round(0).join(d["cluster"].value_counts().rename("posts")))

# REGRESSION: predict log(likes) from audience size and platform (not from comments/shares)
if has("likes", "followers_count", "platform"):
    d = df.dropna(subset=["likes", "followers_count", "platform"]).copy()
    X = pd.get_dummies(d[["platform"]]).join(np.log1p(d["followers_count"]).rename("log_followers"))
    y = np.log1p(d["likes"])
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42)
    print("Test R²:", round(r2_score(yte, LinearRegression().fit(Xtr, ytr).predict(Xte)), 3))

**Exit question:** Which technique would help answer your marketing question, and what evidence would you need?
(For example, classification needs trustworthy labels; with scraped comments you must create them yourself, and
mixed English/Urdu/Roman Urdu text can make English-only sentiment tools unreliable.)

## Step 10 · Package the submission

In [ ]:
summary = pd.DataFrame({
    "item": ["Rows (raw)", "Rows (after cleaning)", "Columns", "Missing cells (after cleaning)", "Duplicate rows (raw)"],
    "value": [len(raw), len(df), df.shape[1], int(df.isna().sum().sum()), int(raw.duplicated().sum())],
})
summary.to_csv("submission/dataset_summary.csv", index=False)
df.to_csv("submission/cleaned_data.csv", index=False)
display(summary)

shutil.make_archive("project1_submission", "zip", "submission")
try:
    from google.colab import files
    files.download("project1_submission.zip")
except ImportError:
    print("Zip created: project1_submission.zip")